# Laptop Market Analysis - Data Processing Pipeline

## 1. Methodology & Execution

This pipeline processes nested, heterogeneous JSON structures scraped from multiple retail websites into a strict, flat schema engineered specifically for Business Intelligence (Tableau, PowerBI).

**Data Engineering Flow:**
- **String Normalization & Cleaning:** Aggressively standardizes Vietnamese terminology (removing linguistic accents, case-folding) to consolidate disparate labels (e.g., `ổ cứng`, `lưu trữ` $\rightarrow$ `storage`).
- **Advanced Regex Heuristics:** Safely parses absolute hardware capacities (RAM, Storage, Battery Wh) via complex negative lookaheads. This prevents dangerous data bleeding (e.g., mistakenly capturing 16MB Cache or 8GB VRAM as System RAM, or extracting 16 inch displays as memory constraints).
- **Categorical Feature Engineering:** Strategically bins continuous values (like placing raw integers into dynamic `price_segment` partitions) and maps complex device hierarchies. Device Categories (Gaming/Office/Creator) are inferred natively from hardware intersection points (e.g., if a system holds $\ge$ 32GB RAM and a Dedicated GPU, it acts as a Creator model; if it's strictly a Dedicated GPU with $\ge$ 120Hz display, it is labeled Gaming).
- **Statistical Imputation (`DataImputer`):** Crucial numerical features natively missing from scrapers (e.g., `battery_wh`, `weight_kg`, `screen_res`) are mathematically derived mimicking actual supply chains. The engine maps strict composite keys `(Brand, Series, Screen Size, Price Tier)` and calculates the statistical **Mode** (the most prevalent configuration of real-world identical siblings) to impute the blank. Unsafe or anomalous hardware structures safely retain pure `NaN` to evade hallucinatory data.

## 2. Global Entity Schema 

The output schema extracts exactly **24 BI-Ready features** constructed concisely without prefixes to optimize dashboard label sizing.

| Field | Purpose & Semantic Mapping |
| :--- | :--- |
| `id`, `source`, `url` | Core metadata. `url` is preserved intrinsically allowing analysts to actively audit Flash Sale outliers directly at the manufacturer's link. |
| `brand`, `name`, `is_available` | Root classifiers. Brands are rigidly standardized (`Hp` $\rightarrow$ `HP`, `Macbook` $\rightarrow$ `Apple`). |
| `category` | Inferred tier grouping (`Gaming`, `Creator`, `Office`) utilized for cross-competition analysis and market density. |
| `is_ai` | Boolean hardware flag signaling edge accelerations (NPU, Copilot+, Snapdragon, Neural Engines) to validate AI-Price valuation. |
| `price`, `price_segment` | Financial constraints partitioned into macro ranges (`Budget` $<15M$, `Mainstream` $15-25M$, `Mid-high` $25-45M$, `Premium` $>45M$). |
| `cpu_brand`, `cpu_family` | Bounded hierarchical grouping (`Core Ultra 7`, `Ryzen AI 9`, `Apple M3 Max`) isolating manufacturer dependence. |
| `gpu_type`, `gpu_model` | Classifies Graphics architecture structures: `Dedicated` (RTX/RX), `Integrated` (Iris/Radeon), or `Unified` (Apple). |
| `ram_gb`, `ram_type` | Strict integer allocations (e.g. $16$, $32$). Reconstructs arrays mathematically (e.g., $16GB \times 2 \rightarrow 32$). Crucial for Obj 5. |
| `storage_gb`, `storage_type` | Validates limits (128GB - 8192GB). Defaults to `SSD` formatting protocols. |
| `battery_wh` | High-fidelity integer mapping `Wh`/`Whr` strings. Imputed across similar chassis sizes to track Office productivity life. |
| `screen_size_inches`, `screen_res`, `screen_hz` | Float representations of visual arrays. Safely converts text (`FHD`, `2.8K`) to analytical strings boundaries. |
| `weight_kg`, `os_family` | Resolves dimensional grams into strict Kg parameters (`0.95`). |
| `avg_rating`, `review_count`, `satisfied_count` | Decodes text magnitude multipliers (`2.5k` $\rightarrow$ $2500$) facilitating quantitative Voice of the Customer (VoC) analytics. |

## Data Loading

In [1]:
import json, os, re, statistics
import pandas as pd
import unicodedata

DATA_DIR = "data"

def load_json_files(data_dir):
    all_data = []
    f_count = 0
    for file in os.listdir(data_dir):
        if file.startswith('data_') and file.endswith('.json'):
            path = os.path.join(data_dir, file)
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    if isinstance(data, list):
                        all_data.extend(data)
                    else:
                        all_data.append(data)
                f_count += 1
            except Exception as e:
                print(f"Err {file}: {e}")
    
    print(f"Loaded: {len(all_data)} records, {f_count} files")
    return all_data

raw_records = load_json_files(DATA_DIR)

Loaded: 3732 records, 7 files


## Text Cleaners & Regex 

In [2]:
def remove_accents(input_str):
    if input_str is None:
        return ""
    # Strip linguistic accents and convert to standard lowercase
    s1 = unicodedata.normalize('NFD', input_str)
    s2 = "".join([c for c in s1 if not unicodedata.combining(c)])
    # Phonetic normalization mapping
    s2 = s2.replace('đ', 'd').replace('Đ', 'D')
    return s2.lower().strip()

def normalize_key(key):
    return remove_accents(key)

In [3]:
def extract_ram_capacity(ram_string):
    if not ram_string: return None
    s = str(ram_string).lower()
    
    # Handle multiplier formats (e.g. 2x8GB, 8GB x 2)
    mul_match1 = re.search(r'(\d+)\s*x\s*(\d+)\s*(?:gb|g|gib)\b', s)
    if mul_match1:
        val = int(mul_match1.group(1)) * int(mul_match1.group(2))
        if val in [4, 8, 12, 16, 24, 32, 40, 48, 64, 96, 128]:
            return val
            
    mul_match2 = re.search(r'(\d+)\s*(?:gb|g|gib)\s*x\s*(\d+)', s)
    if mul_match2:
        val = int(mul_match2.group(1)) * int(mul_match2.group(2))
        if val in [4, 8, 12, 16, 24, 32, 40, 48, 64, 96, 128]:
            return val

    # Match combined configurations (e.g. 8gb onboard + 8gb)
    plus_matches = re.finditer(r'(\d+)\s*(?:gb|g|gib)\b', s)
    vals = [int(m.group(1)) for m in plus_matches if 0 < int(m.group(1)) <= 128]
    
    if "+" in s and len(vals) >= 2:
        sum_val = sum(vals[:2]) 
        if sum_val in [8, 12, 16, 24, 32, 40, 48, 64, 96, 128]:
            return sum_val

    # Match standard capacity patterns
    match = re.search(r'(\d+)\s*(?:gb|g|gib)\b', s)
    if match:
        val = int(match.group(1))
        if val <= 128: return val
    
    # Fallback absolute numerical extraction avoiding external metric capture (inch, hz)
    match2 = re.search(r'\b(4|8|12|16|24|32|48|64|96|128)\b(?!\s*(?:inch|"|\'|hz|k|w))', s)
    if match2:
        return int(match2.group(1))
        
    return None

def extract_ram_type(ram_string):
    if not ram_string: return None
    ram_string = str(ram_string).upper()
    if 'LPDDR5X' in ram_string: return 'LPDDR5X'
    if 'LPDDR5' in ram_string: return 'LPDDR5'
    if 'DDR5' in ram_string: return 'DDR5'
    if 'LPDDR4X' in ram_string: return 'LPDDR4X'
    if 'LPDDR4' in ram_string: return 'LPDDR4'
    if 'DDR4' in ram_string: return 'DDR4'
    if 'UNIFIED' in ram_string or 'M-SERIES' in ram_string: return 'Unified'
    return None

def extract_storage_capacity(storage_string):
    if not storage_string: return None
    
    s = str(storage_string).lower()

    # Utilize TB boundary mapping validation
    for m in re.finditer(r'(\d+)\s*(?:tb|tbp)\b', s):
        cap = int(m.group(1))
        if 1 <= cap <= 8:
            return cap * 1024

    # Capture sequential valid GB capacities via integer bounds
    for m in re.finditer(r'(\d+)\s*(?:gb|g|gib)\b', s):
        val = int(m.group(1))
        if 120 <= val <= 8192: 
            return val
    
    # Fallback to predefined storage configuration vectors
    st2 = re.search(r'\b(128|256|512|1000|1024|2000|2048|4000|4096)\b', s)
    if st2: return int(st2.group(1))
    
    return None

def extract_storage_type(storage_string):
    # Default vector mapping
    if not storage_string: return 'SSD' 
    storage_string = str(storage_string).upper()
    if 'SSD' in storage_string or 'NVME' in storage_string: return 'SSD'
    if 'HDD' in storage_string: return 'HDD'
    if 'EMMC' in storage_string: return 'eMMC'
    return 'SSD'

def extract_screen_size(screen_string):
    if not screen_string: return None
    match = re.search(r'(\d{2}(?:\.\d+)?)\s*(?:\"|inch|\'\')', str(screen_string), re.IGNORECASE)
    if match:
        return float(match.group(1))
    match_start = re.search(r'^(\d{2}(?:\.\d+)?)', str(screen_string))
    if match_start:
        return float(match_start.group(1))
    return None

def extract_screen_hz(screen_string):
    if not screen_string: return None
    match = re.search(r'(\d+)\s*(?:hz)', str(screen_string), re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

def extract_screen_resolution(screen_string):
    if not screen_string: return None
    screen_string = str(screen_string).upper()
    if '4K' in screen_string or '3840' in screen_string or 'UHD+' in screen_string: return '4K'
    if '3K' in screen_string or '2880' in screen_string or '3024' in screen_string or '3456' in screen_string: return '3K'
    if '2.8K' in screen_string or '2880X1800' in screen_string or '2880 X 1800' in screen_string: return '2.8K'
    if '2.5K' in screen_string or 'WQXGA' in screen_string or '2560' in screen_string: return '2.5K'       
    if '2K' in screen_string: return '2K'
    if 'FHD+' in screen_string or '1920 X 1200' in screen_string or '1920X1200' in screen_string or 'WUXGA' in screen_string: return 'FHD+'
    if 'FHD' in screen_string or 'FULL HD' in screen_string or '1920 X 1080' in screen_string or '1920X1080' in screen_string: return 'FHD'
    if re.search(r'\bHD\b', screen_string): return 'HD'
    return None

def extract_gpu_type(gpu_string):
    if not gpu_string: return "Integrated"
    gpu_str_lower = str(gpu_string).lower()
    
    dedicated_keywords = ['rtx', 'gtx', 'rx', 'radeon pro', 'nvdia', 'nvidia', 'geforce', 'arc', 'mx', 'quadro']
    unified_keywords = ['apple', 'unified', 'macbook']

    for kw in dedicated_keywords:
        if kw in gpu_str_lower: return "Dedicated"
    for kw in unified_keywords:
        if kw in gpu_str_lower: return "Unified"
    return "Integrated"

def extract_gpu_model(gpu_raw):
    if not gpu_raw: return "Other"
    
    g_str = str(gpu_raw).upper().replace('™', '').replace('®', '')
    
    # NVIDIA RTX / Quadro
    rtx_match = re.search(r'(RTX\s*(?:PRO\s*)?(?:A)?\d{3,4}(?:\s*TI|\s*SUPER|\s*ADA)?)', g_str)
    if rtx_match: return "NVIDIA " + re.sub(r'\s+', ' ', rtx_match.group(1))
    
    gtx_match = re.search(r'(GTX\s*\d{4}(?:\s*TI)?)', g_str)
    if gtx_match: return "NVIDIA " + re.sub(r'\s+', ' ', gtx_match.group(1))

    quadro_match = re.search(r'(QUADRO\s*[A-Z]?\d{3,4})', g_str)
    if quadro_match: return "NVIDIA " + re.sub(r'\s+', ' ', quadro_match.group(1))
    
    mx_match = re.search(r'(MX\s*\d{3}A?|\d{3}\s*MX)', g_str)
    if mx_match: return "NVIDIA GeForce MX"

    # AMD
    rx_match = re.search(r'(RX\s*\d{4}(?:\s*M|\s*S)?)', g_str)
    if rx_match: return "AMD Radeon " + re.sub(r'\s+', ' ', rx_match.group(1))

    amd_m = re.search(r'(RADEON\s*\d{3}M)', g_str)
    if amd_m: return "AMD " + re.sub(r'\s+', ' ', amd_m.group(1))
    
    if 'RADEON' in g_str: return "AMD Radeon Graphics"

    # QUALCOMM
    if 'ADRENO' in g_str or 'SNAPDRAGON' in g_str: return "Qualcomm Adreno"

    # INTEL
    if 'ARC' in g_str: return "Intel Arc"
    if 'IRIS' in g_str or 'X\\' in g_str or 'XE' in g_str: return "Intel Iris Xe"
    if 'UHD' in g_str: return "Intel UHD"
    if 'HD GRAPHICS' in g_str: return "Intel HD Graphics"

    # APPLE 
    if 'APPLE' in g_str or 'NHÂN GPU' in g_str or 'CORE GPU' in g_str or 'M1' in g_str or 'M2' in g_str or 'M3' in g_str or 'M4' in g_str: 
        return "Apple GPU"

    # Integrated fallback allocation
    if 'INTEL GRAPHIC' in g_str or 'INTEGRATED GRAPHIC' in g_str or 'ONBOARD' in g_str or 'ON BOAD' in g_str or 'SHARED' in g_str: 
        return "Integrated Graphics"
    
    return "Other"

def extract_os_family(os_raw):
    if not os_raw: return "Windows" 
    os_str = str(os_raw).lower()
    if 'windows 11' in os_str or 'win 11' in os_str: return "Windows 11"
    if 'windows 10' in os_str or 'win 10' in os_str: return "Windows 10"
    if 'mac' in os_str: return "macOS"
    if 'dos' in os_str or 'linux' in os_str or 'ubuntu' in os_str: return "DOS/Linux"
    if 'windows' in os_str or 'win' in os_str: return "Windows"
    return "Windows" 

def extract_weight_kg(weight_raw):
    if not weight_raw: return None
    weight_str = str(weight_raw).lower()
    
    # Grammatical pattern validation and float conversion mapping
    mg = re.search(r'(?<!k)(?<!k\s)(?<!\.)(\d{3,4})\s*(?:g|gram)\b', weight_str)
    if mg:
        val = int(mg.group(1))
        if 500 <= val <= 5000:
            return round(val / 1000.0, 2)
            
    match = re.search(r'(\d+[\.,]\d+|\d+)\s*(?:kg)', weight_str, re.IGNORECASE)
    if match:
        val = match.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            return None
    
    m2 = re.search(r'(?:nặng|lượng|weight|khối)[\s\:\-]*(\d+[\.,]\d+|\d+)', weight_str)
    if m2:
        val = m2.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            pass
    return None

def extract_battery_wh(battery_raw):
    if not battery_raw: return None
    s = str(battery_raw).lower()
    
    # Extract explicitly formatted Wh or Whr (e.g., 50 Wh, 42.5 whr, 90 w.h)
    match = re.search(r'(\d+(?:[\.,]\d+)?)\s*(?:wh|whr|w\.h|whrs)\b', s)
    if match:
        val = match.group(1).replace(',', '.')
        try:
            return float(val)
        except ValueError:
            pass
            
    return None

def extract_cpu_family(cpu_raw):
    if not cpu_raw: return "Other"
    cpu_str = str(cpu_raw).lower()

    # Apple M
    apple_match = re.search(r'\b(m[1-5](?:\s*pro|\s*max|\s*ultra)?)\b', cpu_str)
    if apple_match: return f"Apple {apple_match.group(1).title()}"

    # Intel Core Ultra
    ultra_match = re.search(r'ultra\s*(\d)', cpu_str)
    if ultra_match: return f"Core Ultra {ultra_match.group(1)}"

    # Core i-series
    core_match = re.search(r'core\s*(?:i)?\s*(3|5|7|9)', cpu_str)
    if core_match: 
        return f"Core i{core_match.group(1)}"
    else:
        i_match = re.search(r'\b(i[3579])\b', cpu_str)
        if i_match:
            return f"Core {i_match.group(1)}"
    
    if 'pentium' in cpu_str: return "Intel Pentium"
    if 'celeron' in cpu_str: return "Intel Celeron"
    if 'n100' in cpu_str or 'n200' in cpu_str: return "Intel N-series"

    # AMD Ryzen AI
    ryzen_ai_match = re.search(r'ryzen\s*ai\s*(\d)', cpu_str)
    if ryzen_ai_match: return f"Ryzen AI {ryzen_ai_match.group(1)}"

    # AMD Ryzen
    ryzen_match = re.search(r'ryzen\s*(?:z?\d)', cpu_str)
    if ryzen_match:
        match_str = ryzen_match.group(0).title()
        return match_str.replace("Z1", "Z1")
    else:
        r_match = re.search(r'\b(r[3579])\b', cpu_str)
        if r_match:
            return f"Ryzen {r_match.group(1).replace('r', '')}"

    if 'snapdragon' in cpu_str: return "Snapdragon"
    return "Other"

def get_price_segment(price):
    if price is None or price == 0: return None
    if price < 15000000: return "Budget"
    if price <= 25000000: return "Mainstream"
    if price <= 45000000: return "Mid-high"
    return "Premium"

def get_category(name, gpu_type, screen_hz, ram_gb):
    name_lower = str(name).lower() if name else ""

    gaming_keywords = ['gaming', 'nitro', 'tuf', 'rog ', 'legion', 'alienware', 'predator', 'loq', 'ideapad gaming', 'victus', 'cyborg', 'katana', 'bravo', 'stealth', 'raider', 'omen', 'tuf dash']
    creator_keywords = ['creator', 'studio', 'proart', 'conceptd', 'zenbook pro', 'aero', 'macbook pro']
    office_keywords = ['vivobook', 'zenbook', 'expertbook', 'swift', 'aspire', 'ideapad', 'thinkpad', 'thinkbook', 'macbook air', 'elitebook', 'probook', 'pavilion', 'envy', 'hp 14', 'hp 15']    

    # Creator tier hierarchy
    if any(kw in name_lower for kw in creator_keywords): return "Creator"
    
    # Predefined keyword matching arrays
    if any(kw in name_lower for kw in gaming_keywords): return "Gaming"
    if any(kw in name_lower for kw in office_keywords): return "Office"

    # Hardware component inference models
    if ram_gb and ram_gb >= 32 and gpu_type == "Dedicated": return "Creator"
    if gpu_type == "Dedicated" and screen_hz and screen_hz >= 120: return "Gaming"

    return "Office"

def check_ai_chip(cpu_raw, name):
    combine_str = f"{cpu_raw or ''} {name or ''}".lower()
    ai_keywords = ['npu', 'ai boost', 'ultra', 'ryzen ai', 'snapdragon', 'copilot', 'xdna', 'neural engine']
    return any(kw in combine_str for kw in ai_keywords)

def parse_satisfied_count(text):
    if not text: return 0
    match_k = re.search(r'([\d\.,]+)\s*(k|K|triệu)?\s*(khách|lượt)', str(text).lower())
    if match_k:
        num_str = match_k.group(1).replace(',', '.')
        try:
            num = float(num_str)
        except ValueError:
            return 0
        unit = match_k.group(2)
        if unit == 'k': return int(num * 1000)
        elif unit == 'triệu': return int(num * 1000000)
        return int(num)

    match_digit = re.search(r'^\s*([\d\.,]+)', str(text))
    if match_digit:
        num_str = match_digit.group(1).replace(',', '.')
        try:
            return int(float(num_str))
        except ValueError:
            pass
    return 0

## Transformer Class

In [4]:
class LaptopTransformer:
    def __init__(self):
        # Attribute standardization map
        self.key_mapping = {
            'cpu': ['cpu', 'vi xu ly', 'xu ly', 'chip', 'bo xu ly'],
            'ram': ['ram', 'bo nho trong', 'bo nho'],
            'storage': ['o cung', 'rom', 'storage', 'ssd', 'hdd', 'luu tru', 'dung luong'],
            'vga': ['vga', 'card', 'do hoa', 'gpu'], 
            'screen': ['man hinh', 'hien thi', 'display', 'do phan giai', 'tan so', 'tam nen', 'screen'], 
            'os': ['os', 'he dieu hanh', 'operating system'],
            'weight': ['weight', 'khoi luong', 'trong luong', 'kich thuoc', 'nang', 'trong'],
            'battery': ['pin', 'battery', 'dung luong pin']
        }
        # Parameter exclusion filters 
        self.excluded_keys = ['toi da', 'max', 'nang cap', 'up to', 'phu', 'reader', 'the nho']

    def _find_spec(self, raw_specs, spec_type):
        """Aggregate specs using intersection of target and exclusion parameters"""
        target_keys = self.key_mapping.get(spec_type, [])
        found_values = []
        for k, v in raw_specs.items():
            norm_k = normalize_key(k)
            
            # Sub-filter: Exclude graphics capacity strings parsing as system memory
            if spec_type == 'ram' and any(x in norm_k for x in ['do hoa', 'card', 'vga', 'dem', 'cache', 'gpu', 'vram']):
                continue
            
            # Sub-filter: Exclude system memory constraints parsing as storage vectors
            if spec_type == 'storage' and any(x in norm_k for x in ['ram', 'pin', 'the nho']):
                continue
            
            # Match parameters bypassing constraint constants 
            if any(tk in norm_k for tk in target_keys) and not any(ek in norm_k for ek in self.excluded_keys):
                found_values.append(str(v).strip())
        return " | ".join(found_values)

    def process(self, raw_data):
        # Omit irrelevant payload nodes (accessories, empty entities)
        name = raw_data.get('product', {}).get('name', '')
        if not name or "balo" in name.lower() or "chuột" in name.lower() or "túi" in name.lower() or "apple care" in name.lower():
            return None

        current_price = raw_data.get('pricing', {}).get('current_price', 0)     
        is_available = raw_data.get('product', {}).get('is_available', True)    

        if current_price == 0:
            return None

        specs_dict = raw_data.get('specs_raw', {})
        if not specs_dict:
             return None

        brand_match = re.search(r'(Asus|Acer|Dell|HP|Lenovo|MSI|Apple|Macbook|Gigabyte|LG|Microsoft)', name, re.IGNORECASE)
        if brand_match:
            b_name = brand_match.group(1).title()
            brand = "Apple" if b_name == "Macbook" else ("HP" if b_name == "Hp" else ("MSI" if b_name == "Msi" else ("LG" if b_name == "Lg" else b_name)))      
        else:
            brand = "Other"

        # Apply fallback node inspection via native product title matching
        raw_cpu = self._find_spec(specs_dict, 'cpu')
        if not raw_cpu: raw_cpu = name 
        
        raw_ram = self._find_spec(specs_dict, 'ram')
        if not raw_ram: raw_ram = name
        
        raw_storage = self._find_spec(specs_dict, 'storage')
        if not raw_storage: raw_storage = name
        
        raw_vga = self._find_spec(specs_dict, 'vga')
        if not raw_vga: raw_vga = name
        
        raw_screen = self._find_spec(specs_dict, 'screen')
        raw_os = self._find_spec(specs_dict, 'os')
        raw_weight = self._find_spec(specs_dict, 'weight')
        raw_battery = self._find_spec(specs_dict, 'battery')

        # Feature extraction phase
        cpu_family = extract_cpu_family(raw_cpu)
        if cpu_family == "Other" and brand == "Apple":
            m_match = re.search(r'\b(m[1-5](?:\s*pro|\s*max)?)\b', name.lower())
            if m_match:
                cpu_family = f"Apple {m_match.group(1).title()}"

        # CPU model logic mapping
        cpu_str_lower = raw_cpu.lower()
        if "intel" in cpu_str_lower or "core" in cpu_family.lower() or "pentium" in cpu_family.lower() or "celeron" in cpu_family.lower() or "n-series" in cpu_family.lower():
            cpu_brand = "Intel"
        elif "amd" in cpu_str_lower or "ryzen" in cpu_family.lower():
            cpu_brand = "AMD"
        elif "apple" in cpu_str_lower or "m1" in cpu_str_lower or "m2" in cpu_str_lower or "m3" in cpu_str_lower or "m4" in name.lower() or brand == "Apple" or "apple" in cpu_family.lower():   
            cpu_brand = "Apple"
        elif "snapdragon" in cpu_str_lower or "snapdragon" in cpu_family.lower():
            cpu_brand = "Qualcomm"
        else:
            cpu_brand = "Other"
            
        is_ai = check_ai_chip(raw_cpu + str(specs_dict), name)

        gpu_model = extract_gpu_model(raw_vga)
        if brand == "Apple" and gpu_model == "Other":
            gpu_model = "Apple GPU"

        gpu_type = extract_gpu_type(raw_vga)
        if gpu_type == "Integrated" and gpu_model != "Other":
            if any(key in gpu_model.upper() for key in ['RTX', 'GTX', ' RX ', 'QUADRO', 'ARC A']):
                gpu_type = "Dedicated"
            elif "APPLE" in gpu_model.upper():
                gpu_type = "Unified"

        ram_gb = extract_ram_capacity(raw_ram)
        ram_type = extract_ram_type(raw_ram)
        
        storage_gb = extract_storage_capacity(raw_storage)
        storage_type = extract_storage_type(raw_storage)

        # M-Series core architecture constant assignments
        if brand == "Apple": 
            if not storage_type and storage_gb: storage_type = "SSD"
            if not ram_type and "Apple M" in cpu_family: ram_type = "Unified"

        screen_size = extract_screen_size(raw_screen)
        screen_hz = extract_screen_hz(raw_screen)
        screen_res = extract_screen_resolution(raw_screen)
        
        os_family = extract_os_family(raw_os)
        if brand == "Apple": os_family = "macOS" 
        
        weight_kg = extract_weight_kg(raw_weight)
        battery_wh = extract_battery_wh(raw_battery)

        cat = get_category(name, gpu_type, screen_hz, ram_gb)

        master_json = {
            "id": raw_data.get("id"),
            "source": raw_data.get("metadata", {}).get("source"),
            "url": raw_data.get("metadata", {}).get("url"),
            
            "brand": brand,
            "name": name,
            "category": cat,
            "is_ai": is_ai,
            "is_available": is_available,
            
            "price": current_price,
            "price_segment": get_price_segment(current_price),

            "cpu_brand": cpu_brand,
            "cpu_family": cpu_family,

            "gpu_type": gpu_type,
            "gpu_model": gpu_model,
            
            "ram_gb": ram_gb,
            "ram_type": ram_type,

            "storage_gb": storage_gb,
            "storage_type": storage_type,

            "screen_size_inches": screen_size,
            "screen_res": screen_res,
            "screen_hz": screen_hz,

            "os_family": os_family,
            "weight_kg": weight_kg,
            "battery_wh": battery_wh,
            
            "avg_rating": raw_data.get("social", {}).get("avg_rating", 0.0),      
            "review_count": raw_data.get("social", {}).get("review_count", 0),        
            "satisfied_count": parse_satisfied_count(raw_data.get("social", {}).get("satisfied_info", ""))
        }

        return master_json

## ETL Data Processor

In [5]:
import pandas as pd
import json
import os
import statistics

class DataImputer:
    """Missing data interpolation engine utilizing contextual similarity metrics"""
    def __init__(self, dataset):
        self.dataset = dataset
        self.stats = {
            'weight': 0, 'hz': 0, 'ram_type': 0, 
            'ram_cap': 0, 'storage': 0, 'res': 0, 'battery': 0
        }
        
        # Initialize logical parameters
        for row in self.dataset:
            row['_series'] = self._get_series(row.get('name', ''))
            
            price_seg = row.get('price_segment')
            if price_seg is None or price_seg == "":
                row['_price_seg'] = 'Unknown'
            else:
                row['_price_seg'] = price_seg

        self.pools = {}
        
        # Mapping matrices definition arrays
        self.pools['weight_strict'] = self._build_pool('weight_kg', ['brand', '_series', 'screen_size_inches', 'gpu_model'])
        self.pools['weight_office'] = self._build_pool('weight_kg', ['brand', '_series', 'screen_size_inches'], allow_other=True)
        self.pools['battery'] = self._build_pool('battery_wh', ['brand', '_series', 'screen_size_inches', 'category'], allow_other=True)
        self.pools['ram_type'] = self._build_pool('ram_type', ['brand', 'cpu_family'], allow_other=True)
        self.pools['res'] = self._build_pool('screen_res', ['brand', '_series', 'screen_size_inches', '_price_seg'], allow_other=True)
        self.pools['hz'] = self._build_pool('screen_hz', ['brand', '_series', 'screen_size_inches', 'category'], allow_other=True)
        self.pools['ram_cap'] = self._build_pool('ram_gb', ['brand', '_series', 'cpu_family', '_price_seg'], allow_other=True)
        self.pools['storage'] = self._build_pool('storage_gb', ['brand', '_series', 'cpu_family', '_price_seg'], allow_other=True)

    def _get_series(self, name):
        """Extract primary product series identifier"""
        series_list = [
            'macbook air', 'macbook pro', 'tuf', 'rog', 'zephyrus', 'vivobook', 'zenbook', 'expertbook', 
            'nitro', 'predator', 'aspire', 'swift', 'spin', 'conceptd', 'legion', 'ideapad', 'thinkpad', 
            'thinkbook', 'loq', 'yoga', 'alienware', 'inspiron', 'vostro', 'xps', 'latitude', 'precision',
            'victus', 'omen', 'pavilion', 'envy', 'spectre', 'probook', 'elitebook', 'bravo', 'katana', 
            'cyborg', 'stealth', 'raider', 'titan', 'modern', 'prestige', 'summit'
        ]
        
        name_lower = str(name).lower()
        for series in series_list:
            if series in name_lower:
                return series
        return "Unknown"

    def _build_pool(self, target_key, keys, allow_other=False):
        """Aggregate statistical mode against bounded feature combinations"""
        pools_groups = {}
        for row in self.dataset:
            val = row.get(target_key)
            if val is None or pd.isna(val):
                continue
            
            sig = tuple(row.get(k) for k in keys)
            
            if any(not v or v == "Unknown" for v in sig):
                continue
                
            if not allow_other and any(str(v) == "Other" for v in sig):
                continue
            
            if sig not in pools_groups:
                pools_groups[sig] = []
            pools_groups[sig].append(val)
            
        inferences = {}
        for sig, vals in pools_groups.items():
            try: 
                inferences[sig] = statistics.mode(vals)
            except statistics.StatisticsError: 
                inferences[sig] = vals[0]
                
        return inferences

    def _assign(self, row, key, pool_key, sig, stat_key, fallback=None):
        if not row.get(key):
            if sig in self.pools[pool_key]:
                row[key] = self.pools[pool_key][sig]
                self.stats[stat_key] += 1
            elif fallback is not None:
                row[key] = fallback
                self.stats[stat_key] += 1

    def run(self):
        for row in self.dataset:
            brand = row.get('brand')
            series = row.get('_series')
            screen_size = row.get('screen_size_inches')
            cpu = row.get('cpu_family')
            category = row.get('category')
            price_seg = row.get('_price_seg')
            gpu = row.get('gpu_model')
            name_str = str(row.get('name', '')).lower()
            
            self._assign(row, 'weight_kg', 'weight_strict', (brand, series, screen_size, gpu), 'weight')
            if category != "Gaming": 
                self._assign(row, 'weight_kg', 'weight_office', (brand, series, screen_size), 'weight')
            
            self._assign(row, 'battery_wh', 'battery', (brand, series, screen_size, category), 'battery')
            self._assign(row, 'screen_res', 'res', (brand, series, screen_size, price_seg), 'res')
            
            # Framework logical fallbacks implementation
            fallback_hz = None
            if category == "Office" and price_seg in ["Budget", "Mainstream"]: 
                fallback_hz = 60
            elif brand == "Apple": 
                is_pro_m_series = "pro" in name_str and any(m in name_str for m in ["m1", "m2", "m3", "m4"])
                fallback_hz = 120 if is_pro_m_series else 60
                    
            self._assign(row, 'screen_hz', 'hz', (brand, series, screen_size, category), 'hz', fallback_hz)

            self._assign(row, 'ram_type', 'ram_type', (brand, cpu), 'ram_type')
            self._assign(row, 'ram_gb', 'ram_cap', (brand, series, cpu, price_seg), 'ram_cap')
            self._assign(row, 'storage_gb', 'storage', (brand, series, cpu, price_seg), 'storage')

        for row in self.dataset:
            row.pop('_series', None)
            row.pop('_price_seg', None)
            
        print(f"Imputed: {self.stats}")
        return self.dataset

transformer = LaptopTransformer()
clean_data = []

for raw in raw_records:
    try:
        if (p := transformer.process(raw)):
            clean_data.append(p)
    except Exception as e:
        print(f"Err obj {raw.get('id')}: {e}")

imputer = DataImputer(clean_data)
final_data = imputer.run()

print(f"Total processed: {len(final_data)}")

# Export to JSON
out_path_json = os.path.join(DATA_DIR, "transformed_data.json")
with open(out_path_json, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

# Export to CSV
out_path_csv = os.path.join(DATA_DIR, "transformed_data.csv")
df_export = pd.DataFrame(final_data)
df_export.to_csv(out_path_csv, index=False, encoding='utf-8-sig')

Imputed: {'weight': 985, 'hz': 1208, 'ram_type': 423, 'ram_cap': 139, 'storage': 255, 'res': 303, 'battery': 725}
Total processed: 2798


## Missing Data Statistics

In [6]:
import pandas as pd

df = pd.DataFrame(final_data)
total = len(df)

missing = df.isnull().sum().to_frame('Missing_Count')
missing['Missing_Percentage'] = (missing['Missing_Count'] / total) * 100

# Filter & sort missing values
missing = missing[missing['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
missing['Missing_Percentage'] = missing['Missing_Percentage'].apply(lambda x: f"{x:.2f}%")

print(f"Stats on {total} rows. Missing data:")
missing

Stats on 2798 rows. Missing data:


,Missing_Count,Missing_Percentage
weight_kg,423,15.12%
battery_wh,263,9.40%
screen_hz,176,6.29%
screen_res,140,5.00%
storage_gb,115,4.11%
ram_gb,98,3.50%
screen_size_inches,16,0.57%
ram_type,14,0.50%
